# Walmart Sales Forecasting — TimesFM

| Field | Details |
|---|---|
| **მოდელი** | TimesFM (Time Series Foundation Model) |
| **კატეგორია** | Foundation Model — Pretrained, Zero-shot |
| **შემქმნელი** | Google Research |
| **Logging** | WandB · project: `walmart-forecasting` |

>  **Runtime → Change runtime type → T4 GPU** (inference-ისთვის GPU სჭირდება)

---

## რა არის Foundation Model?

წინასწარ გაწვრთნილი მოდელი, რომელიც გამოიყენება მრავალი ამოცანისთვის — ხშირად zero-shot, fine-tuning-ის გარეშე.

| Domain | მაგალითები |
|---|---|
| **NLP** | GPT-3, LLaMA — pretrained on text |
| **Vision** | CLIP, DINO — pretrained on images |
| **Time Series** | **TimesFM**, Chronos, Moirai — pretrained on time series |

---

## რატომ TimesFM?

გამოქვეყნდა 2024 წელს: *"A decoder-only foundation model for time-series forecasting"*.  
წინასწარ გაწვრთნილია **100 მილიარდზე მეტ** რეალურ დროით წერტილზე (ფინანსური, retail, ამინდი, IoT).

### მთავარი თვისებები

- **Zero-shot forecasting** — fine-tuning არ სჭირდება; პირდაპირ ვაწვდით მონაცემს და ვიღებთ პროგნოზს.
- **Multi-domain pretraining** — მოდელს უკვე ნანახი აქვს retail, ამინდისა და IoT მონაცემები.
- **Cross-frequency support** — საათობრივი / დღიური / კვირეული / თვიური frequency flags.

### არქიტექტურა

| კომპონენტი | დეტალი |
|---|---|
| არქიტექტურა | Decoder-only Transformer (GPT-ის მსგავსი) |
| პარამეტრები | ~200 მილიონი |
| Input | წარსული კონტექსტი (მაქს. **512 timestep**) |
| Output | Multi-step forecast |

---

## Experiment Runs

| Run | აღწერა |
|---|---|
| `TimesFM_ZeroShot` | Pretrained მოდელი პირდაპირ ვალიდაციაზე — **fine-tuning გარეშე**, 128-კვირიანი კონტექსტი |
| `TimesFM_LongContext` | გრძელი კონტექსტი (**256 კვირა**) |
| `TimesFM_Final` | საუკეთესო კონფიგურაცია, გაშვებული სრულ test.csv-ის ჰორიზონტზე |

---

## რას ველოდებით?

| | |
|---|---|
| **Zero-shot performance** | შეიძლება კარგი იყოს, შეიძლება საშუალო |
| **vs. trained models** | ჩვენ მიერ გაწვრთნილ მოდელებთან კონკურენცია რთულია |
| **საინტერესო კითხვა** | რამდენად კარგად ახდენს გენერალიზაციას კონკრეტულად retail მონაცემებზე? |

## 1. Setup

In [ ]:
# TimesFM ინსტალაცია — Google-ის ოფიციალური repo
!pip install timesfm wandb --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import wandb

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'

import os
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Data: {DATA_DIR}")

In [ ]:
# WandB login
from google.colab import userdata

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("WandB logged in")
except Exception as e:
    wandb.login()

WANDB_PROJECT = "walmart-forecasting"
print(f"WandB project: {WANDB_PROJECT}")

## 2. TimesFM მოდელის ჩატვირთვა

TimesFM Google-ის HuggingFace repo-იდან ჩავტვირთავთ. მოდელი ~200M parameters, ~800MB size.

**შენიშვნა:** პირველად ჩატვირთვისას რამდენიმე წუთი შეიძლება დასჭირდეს — მოდელი HuggingFace-იდან იტვირთება.

`max_horizon`-ს ვაყენებთ 40-ზე (და არა 12-ზე) — test.csv-ში ყველაზე გრძელი სერიისთვის საჭირო prediction horizon 39 კვირაა, ხოლო validation-ისთვის მხოლოდ პირველ 12 ნაბიჯს გამოვიყენებთ.

In [ ]:
import timesfm

MODEL_ID = "google/timesfm-2.5-200m-pytorch"
MAX_HORIZON = 40

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(MODEL_ID)

tfm.compile(
    timesfm.ForecastConfig(
        max_context=512,
        max_horizon=MAX_HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM loaded successfully")

## 3. მონაცემები + Long-format

In [ ]:
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')

train_raw['Date'] = pd.to_datetime(train_raw['Date'])
test_raw['Date'] = pd.to_datetime(test_raw['Date'])

print(f"Train: {train_raw.shape}")
print(f"Test:  {test_raw.shape}")
print(f"Train range: {train_raw['Date'].min().date()} → {train_raw['Date'].max().date()}")
print(f"Test range:  {test_raw['Date'].min().date()} → {test_raw['Date'].max().date()}")

In [ ]:
# Long-format transformation (როგორც N-BEATS/PatchTST/DLinear-ისთვის)
def to_timesfm_format(df, has_target=True):
    df = df.copy()
    df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
    df = df.rename(columns={'Date': 'ds'})
    if has_target:
        df = df.rename(columns={'Weekly_Sales': 'y'})
        return df[['unique_id', 'ds', 'y']]
    return df[['unique_id', 'ds']]


# სრული ისტორია — testზე საბოლოო inference-ისთვის (filter არ გვჭირდება,
# TimesFM-ს zero-shot ნებისმიერი სიგრძის context-ისთვის შეუძლია მუშაობა)
train_tfm_full = to_timesfm_format(train_raw, has_target=True).sort_values(['unique_id', 'ds']).reset_index(drop=True)
test_tfm = to_timesfm_format(test_raw, has_target=False)

# ფილტრირებული ვერსია — მხოლოდ backtest split-ისთვის, სადაც საჭიროა
# საკმარისი სიგრძე, რომ დარჩეს train ნაწილიც val ჰორიზონტის მოჭრის შემდეგ
MIN_LENGTH = 80
series_lengths = train_tfm_full.groupby('unique_id').size()
valid_ids = series_lengths[series_lengths >= MIN_LENGTH].index
train_tfm = train_tfm_full[train_tfm_full['unique_id'].isin(valid_ids)].reset_index(drop=True)

print(f"სერიები სრულ train-ში: {train_tfm_full['unique_id'].nunique()}")
print(f"სერიები backtest-ისთვის (>= {MIN_LENGTH} კვირა): {train_tfm['unique_id'].nunique()}")
print(f"სერიები test.csv-ში: {test_tfm['unique_id'].nunique()}")

missing_from_train = set(test_tfm['unique_id']) - set(train_tfm_full['unique_id'])
print(f"test-ში მყოფი store/dept, რომელიც train-ში საერთოდ არ არის: {len(missing_from_train)}")

## 4. Train/Val Split + WMAE Metric

In [ ]:
def wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


VAL_HORIZON = 12
train_tfm_sorted = train_tfm.sort_values(['unique_id', 'ds']).reset_index(drop=True)


def per_series_split(df, val_h):
    train_parts, val_parts = [], []
    for uid, group in df.groupby('unique_id'):
        group = group.sort_values('ds')
        if len(group) > val_h:
            train_parts.append(group.iloc[:-val_h])
            val_parts.append(group.iloc[-val_h:])
    return pd.concat(train_parts).reset_index(drop=True), pd.concat(val_parts).reset_index(drop=True)


train_split, val_split = per_series_split(train_tfm_sorted, VAL_HORIZON)

val_dates = val_split['ds'].unique()
train_raw_val_period = train_raw[train_raw['Date'].isin(val_dates)]

print(f"Train split: {train_split.shape}")
print(f"Val split:   {val_split.shape}")

## 5. Run 1 — `TimesFM_ZeroShot`

Zero-shot forecasting — არავითარი fine-tuning, უბრალოდ context-ს ვაძლევთ და forecast-ს ვიღებთ.

**Context length:** 128 კვირა (~2.5 წელი)

`forecast_for_dates` თითოეული სერიისთვის საჭირო prediction თარიღებს პირდაპირ სამიზნე dataframe-იდან იღებს (და არა `date_range`-ით გამოთვლილი დღეებიდან) — ზოგიერთ store/dept-ს გამოტოვებული კვირები აქვს ისტორიაში, ამიტომ თარიღების დაგენერირება მიმდევრობით მცდარ alignment-ს იძლეოდა backtest-ში.

In [ ]:
def forecast_for_dates(tfm_model, train_df, target_df, horizon_cap, context_len):
    """ყოველი unique_id-სთვის ვიღებთ ზუსტად იმ თარიღებს, რაც target_df-შია
    (და არა date_range-ით გამოთვლილს), რომ backtest/test alignment ყოველთვის ზუსტი იყოს."""
    target_dates_map = target_df.sort_values('ds').groupby('unique_id')['ds'].apply(list).to_dict()

    forecasts_list = []
    fallback_ids = []

    for uid, future_dates in target_dates_map.items():
        horizon = min(len(future_dates), horizon_cap)
        series_data = train_df[train_df['unique_id'] == uid].sort_values('ds')

        if len(series_data) == 0:
            fallback_ids.append(uid)
            continue

        context = series_data['y'].values[-context_len:]

        try:
            point_forecast, quantile_forecast = tfm_model.forecast(
                horizon=horizon,
                inputs=[context.astype(np.float32)],
            )
            preds = point_forecast[0][:horizon]
        except Exception as e:
            fallback_ids.append(uid)
            continue

        for date, pred in zip(future_dates[:horizon], preds):
            forecasts_list.append({
                'unique_id': uid,
                'ds': date,
                'TimesFM': float(pred)
            })

    return pd.DataFrame(forecasts_list), fallback_ids


def evaluate_forecast(forecasts_df, val_df, train_raw_val):
    merged = forecasts_df.merge(val_df, on=['unique_id', 'ds'], how='inner')

    train_raw_val_lookup = train_raw_val.copy()
    train_raw_val_lookup['unique_id'] = (
        train_raw_val_lookup['Store'].astype(str) + '_' + train_raw_val_lookup['Dept'].astype(str)
    )
    train_raw_val_lookup = train_raw_val_lookup.rename(columns={'Date': 'ds'})

    merged = merged.merge(
        train_raw_val_lookup[['unique_id', 'ds', 'IsHoliday']],
        on=['unique_id', 'ds'], how='left'
    )

    coverage = len(merged) / len(val_df)
    if coverage < 0.99:
        print(f"გაფრთხილება: validation coverage მხოლოდ {coverage:.1%}-ია ({len(merged)}/{len(val_df)})")

    weights = np.where(merged['IsHoliday'] == True, 5, 1)
    return wmae(merged['y'].values, merged['TimesFM'].values, weights)

In [ ]:
CONTEXT_LEN_ZEROSHOT = 128

wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TimesFM_ZeroShot",
    config={
        'context_len': CONTEXT_LEN_ZEROSHOT,
        'horizon_len': VAL_HORIZON,
        'model': MODEL_ID,
        'frequency': 'weekly',
        'training': 'none (zero-shot)',
    },
    reinit=True,
    tags=['timesfm', 'zeroshot']
)

print(f"Running TimesFM zero-shot on {train_split['unique_id'].nunique()} series...")
forecasts_zeroshot, fallback_zeroshot = forecast_for_dates(tfm, train_split, val_split, VAL_HORIZON, CONTEXT_LEN_ZEROSHOT)

val_wmae_zeroshot = evaluate_forecast(forecasts_zeroshot, val_split, train_raw_val_period)

wandb.log({'val_wmae': val_wmae_zeroshot})
wandb.summary['val_wmae'] = val_wmae_zeroshot
wandb.summary['n_series_forecasted'] = forecasts_zeroshot['unique_id'].nunique()
wandb.summary['n_fallback_series'] = len(fallback_zeroshot)
print(f"\nZero-shot Val WMAE: {val_wmae_zeroshot:.2f}")

wandb.finish()

## 6. Run 2 — `TimesFM_LongContext`

უფრო გრძელი context — 256 კვირა (~5 წელი)

In [ ]:
CONTEXT_LEN_LONG = 256

wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TimesFM_LongContext",
    config={
        'context_len': CONTEXT_LEN_LONG,
        'horizon_len': VAL_HORIZON,
        'model': MODEL_ID,
        'frequency': 'weekly',
    },
    reinit=True,
    tags=['timesfm', 'long_context']
)

print(f"Running TimesFM long context on {train_split['unique_id'].nunique()} series...")
forecasts_long, fallback_long = forecast_for_dates(tfm, train_split, val_split, VAL_HORIZON, CONTEXT_LEN_LONG)

val_wmae_long = evaluate_forecast(forecasts_long, val_split, train_raw_val_period)

wandb.log({'val_wmae': val_wmae_long})
wandb.summary['val_wmae'] = val_wmae_long
wandb.summary['n_fallback_series'] = len(fallback_long)
print(f"\nLong Context Val WMAE: {val_wmae_long:.2f}")

wandb.finish()

## 7. Run 3 — `TimesFM_Final`

საუკეთესო context length-ის შერჩევა და **test.csv-ის სრულ ჰორიზონტზე** (თითოეული store/dept-ისთვის ზუსტად იმდენ კვირაზე, რამდენიც test.csv-ში სჭირდება — მაქს. 39 კვირა) ინფერენსი მთელ ხელმისაწვდომ სატრენინგო მონაცემზე.

Store/dept კომბინაციები, რომლებიც test.csv-ში გვხვდება, მაგრამ train-ში ან საერთოდ არ არსებობს, ან TimesFM-მა ვერ იწინასწარმეტყველა, fallback-ით ივსება — შესაბამისი დეპარტამენტის (Dept) მედიანური კვირეული გაყიდვებით მთელ ქსელში.

In [ ]:
run_results = {
    'zeroshot_128': val_wmae_zeroshot,
    'long_context_256': val_wmae_long,
}
best_run = min(run_results, key=run_results.get)
print(f"Best config: {best_run} (WMAE={run_results[best_run]:.2f})")

BEST_CONTEXT = {
    'zeroshot_128': CONTEXT_LEN_ZEROSHOT,
    'long_context_256': CONTEXT_LEN_LONG,
}[best_run]

In [ ]:
wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TimesFM_Final",
    config={
        'context_len': BEST_CONTEXT,
        'horizon_len': 'per-series (matches test.csv, up to 39)',
        'model': MODEL_ID,
        'frequency': 'weekly',
        'best_config_from_experiments': best_run,
    },
    reinit=True,
    tags=['timesfm', 'final']
)

# Final = სრული train history → test.csv-ის ზუსტი თარიღების პროგნოზი
print(f"Generating final test forecasts on full train history...")
forecasts_final, fallback_uids = forecast_for_dates(tfm, train_tfm_full, test_tfm, MAX_HORIZON, BEST_CONTEXT)

print(f"TimesFM-ით დაფარული სერიები: {forecasts_final['unique_id'].nunique()}")
print(f"Fallback-ზე გადამისამართებული სერიები: {len(fallback_uids)}")

# Fallback: დეპარტამენტის მედიანა მთელ ქსელში (store დაიხურა/ისტორია არასაკმარისია
# ან TimesFM-მა ვერ დააგენერირა პროგნოზი)
dept_medians = train_raw.groupby('Dept')['Weekly_Sales'].median()
global_median = train_raw['Weekly_Sales'].median()

fallback_rows = []
for uid in fallback_uids:
    dept = int(uid.split('_')[1])
    fallback_value = dept_medians.get(dept, global_median)
    fallback_dates = test_tfm.loc[test_tfm['unique_id'] == uid, 'ds'].tolist()
    for date in fallback_dates:
        fallback_rows.append({'unique_id': uid, 'ds': date, 'TimesFM': float(fallback_value)})

forecasts_final = pd.concat([forecasts_final, pd.DataFrame(fallback_rows)], ignore_index=True)

print(f"\nსაბოლოო forecasts shape: {forecasts_final.shape}")
print(f"სულ სერიები (TimesFM + fallback): {forecasts_final['unique_id'].nunique()}")

wandb.summary['best_val_wmae'] = run_results[best_run]
wandb.summary['n_series_final'] = forecasts_final['unique_id'].nunique()
wandb.summary['n_fallback_series_final'] = len(fallback_uids)
wandb.finish()

## 8. Predictions შენახვა

TimesFM მოდელი უკვე წინასწარ გაწვრთნილია — შენახვას არ საჭიროებს. ვინახავთ predictions-ს, და ცალკე — Kaggle submission ფორმატში (`Id, Weekly_Sales`), test.csv-ის ზუსტი row-order-ით.

In [ ]:
forecasts_final.to_csv(f'{MODELS_DIR}/timesfm_forecasts.csv', index=False)
print(f"Forecasts saved: {MODELS_DIR}/timesfm_forecasts.csv")

# Kaggle submission ფორმატი — test.csv-ის row-order-ს ვეყრდნობით,
# რომ ყოველ test row-ს ჰქონდეს ზუსტად ერთი შესაბამისი prediction
submission = test_tfm.merge(forecasts_final, on=['unique_id', 'ds'], how='left')
assert submission['TimesFM'].isna().sum() == 0, "ზოგიერთ test row-ს პროგნოზი აკლია"
assert len(submission) == len(test_raw)

submission_out = pd.DataFrame({
    'Id': test_raw['Store'].astype(str) + '_' + test_raw['Dept'].astype(str) + '_' + test_raw['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': submission['TimesFM'].values
})
submission_out.to_csv(f'{MODELS_DIR}/submission_timesfm.csv', index=False)
print(f"Submission saved: {MODELS_DIR}/submission_timesfm.csv")
print(f"Submission shape: {submission_out.shape} (test.csv: {test_raw.shape[0]} rows)")

# WandB Artifact
wandb.finish() if wandb.run else None
run = wandb.init(project=WANDB_PROJECT, name="TimesFM_ArtifactUpload", reinit=True)

artifact = wandb.Artifact(name="timesfm_forecasts", type="predictions")
artifact.add_file(f'{MODELS_DIR}/timesfm_forecasts.csv')
artifact.add_file(f'{MODELS_DIR}/submission_timesfm.csv')

run.log_artifact(artifact)
print("Forecasts uploaded to WandB Artifacts")

wandb.finish()

## 9. პროგნოზების ვიზუალიზაცია

In [ ]:
sample_ids = forecasts_final['unique_id'].unique()[:3]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, uid in zip(axes, sample_ids):
    hist = train_tfm_full[train_tfm_full['unique_id'] == uid].tail(52)
    fc = forecasts_final[forecasts_final['unique_id'] == uid]

    ax.plot(hist['ds'], hist['y'], label='Historical', color='steelblue')
    ax.plot(fc['ds'], fc['TimesFM'], label='TimesFM Forecast (zero-shot)',
            color='purple', linestyle='--')
    ax.set_title(f'Series: {uid}')
    ax.legend()

plt.tight_layout()
plt.show()

## 10. შეჯამება

In [ ]:
print("TimesFM-ის Zero-shot პროგნოზირება Walmart-ის მონაცემებზე")
print("=" * 60)
print(f"Zero-shot (128 კვირიანი კონტექსტი) — Val WMAE: {val_wmae_zeroshot:.2f}")
print(f"Long context (256 კვირიანი კონტექსტი) — Val WMAE: {val_wmae_long:.2f}")
print(f"საუკეთესო კონფიგურაცია: {best_run} (context={BEST_CONTEXT})")
print(f"Fallback-ზე გადამისამართებული სერიები საბოლოო test-ში: {len(fallback_uids)} / {test_tfm['unique_id'].nunique()}")
print()
print(f"submission_timesfm.csv → {test_raw.shape[0]} row, სრულად test.csv-ის Id-ებზე alignment-ით")

### TimesFM-ის უპირატესობები

| უპირატესობა | აღწერა |
|---|---|
| **Zero-shot** | ტრენინგი საჭირო არ არის |
| **Multi-domain pretraining** | ნანახი აქვს retail, ფინანსების, ამინდისა და IoT მონაცემები |
| **Cross-frequency** | საათობრივი, დღიური, კვირეული და თვიური მონაცემები ერთი მოდელით |
| **Instant deployment** | წვრთნის დრო = 0 |

### შეზღუდვები

| შეზღუდვა | აღწერა |
|---|---|
| **Domain-specific მოდელები** | XGBoost და LightGBM Walmart-ისთვის ხშირად მაინც უკეთესია — feature engineering და tree-based მოდელების უპირატესობა სტრუქტურირებულ მონაცემებზე აშკარაა |
| **მოდელის ზომა** | 200M პარამეტრი vs XGBoost-ის ~1M — inference-ის ეტაპზე გამოთვლითი ღირებულება მაღალია |
| **მოკლე/გაწყვეტილი სერიები** | Store/dept კომბინაციები მცირე ისტორიით ან test-ში ახლად გამოჩენილი — fallback-ს საჭიროებს |